# Entrenar modelo de cinturón
Antes: en Drive crea la carpeta `P3_cinturon` y sube ahí `seatbelt.v1i.folder.zip`.
Menú **Entorno de ejecución → Cambiar tipo de entorno → GPU T4**.

## 1. Configuración
Esta es la única celda que se cambia.

In [ ]:
EPOCAS = 30  #@param {type:"integer"}
BATCH = 32  #@param {type:"integer"}
PACIENCIA = 10  #@param {type:"integer"}
SEMILLA = 7  #@param {type:"integer"}
UMBRAL = 0.7  #@param {type:"number"}
N_EJEMPLOS = 6  #@param {type:"integer"}
CARPETA = "/content/drive/MyDrive/P3_cinturon"  #@param {type:"string"}

## 2. Conectar Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
print(os.listdir(CARPETA))

## 3. Descomprimir las fotos

In [ ]:
import zipfile, glob
zipfile.ZipFile(f'{CARPETA}/seatbelt.v1i.folder.zip').extractall('/content/datos')
if os.path.exists(f'{CARPETA}/propias.zip'):
    zipfile.ZipFile(f'{CARPETA}/propias.zip').extractall('/content/datos/propias')
fotos = []
for ruta in glob.glob('/content/datos/**/*.jpg', recursive=True):
    carpeta = os.path.basename(os.path.dirname(ruta))
    if carpeta == 'seatbelt':
        fotos.append((ruta, 1))
    if carpeta == 'no_seatbelt':
        fotos.append((ruta, 0))
print('Total fotos:', len(fotos))

## 4. Separar entrenamiento (80%) y prueba (20%) por video

In [ ]:
import random
def video(ruta):
    return os.path.basename(ruta).split('_mp4')[0].split('.rf.')[0]
videos = sorted(set(video(r) for r, _ in fotos))
random.seed(SEMILLA)
random.shuffle(videos)
videos_prueba = set(videos[:len(videos) // 5])
entreno = [(r, e) for r, e in fotos if video(r) not in videos_prueba]
prueba = [(r, e) for r, e in fotos if video(r) in videos_prueba]
print('Entrenamiento:', len(entreno), ' Prueba:', len(prueba))

## 5. Cargar las fotos en 224x224

In [ ]:
import tensorflow as tf
def cargar(ruta, etiqueta):
    img = tf.io.decode_jpeg(tf.io.read_file(ruta), channels=3)
    img = tf.image.resize(img, (224, 224))
    return img, etiqueta
def conjunto(lista, mezclar):
    rutas = [r for r, _ in lista]
    etiquetas = [float(e) for _, e in lista]
    ds = tf.data.Dataset.from_tensor_slices((rutas, etiquetas))
    if mezclar:
        ds = ds.shuffle(2000, seed=SEMILLA)
    return ds.map(cargar).batch(BATCH)
ds_entreno = conjunto(entreno, True)
ds_prueba = conjunto(prueba, False)

## 6. Crear el modelo

In [ ]:
base = tf.keras.applications.MobileNetV2(input_shape=(224, 224, 3), include_top=False, weights='imagenet')
base.trainable = False
modelo = tf.keras.Sequential([
    tf.keras.Input((224, 224, 3)),
    tf.keras.layers.RandomRotation(0.05),
    tf.keras.layers.RandomZoom(0.15),
    tf.keras.layers.RandomBrightness(0.3),
    tf.keras.layers.Rescaling(1 / 127.5, offset=-1),
    base,
    tf.keras.layers.GlobalAveragePooling2D(),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(1, activation='sigmoid'),
])
modelo.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])
modelo.summary()

## 7. Entrenar

In [ ]:
parar = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=PACIENCIA, restore_best_weights=True)
historial = modelo.fit(ds_entreno, validation_data=ds_prueba, epochs=EPOCAS, callbacks=[parar])

## 8. Gráficas del entrenamiento

In [ ]:
import matplotlib.pyplot as plt
h = historial.history
plt.figure(figsize=(12, 4))
plt.subplot(1, 2, 1)
plt.plot(h['accuracy'], label='entrenamiento')
plt.plot(h['val_accuracy'], label='prueba')
plt.title('Exactitud')
plt.xlabel('época')
plt.legend()
plt.subplot(1, 2, 2)
plt.plot(h['loss'], label='entrenamiento')
plt.plot(h['val_loss'], label='prueba')
plt.title('Error (loss)')
plt.xlabel('época')
plt.legend()
plt.show()

## 9. Nota del modelo y matriz de confusión

In [ ]:
import numpy as np
reales = np.array([e for _, e in prueba])
probabilidades = modelo.predict(ds_prueba, verbose=0)[:, 0]
predichos = (probabilidades > UMBRAL).astype(int)
print('Exactitud total:', round((reales == predichos).mean(), 3))
print('Con cinturón bien detectado:', round(predichos[reales == 1].mean(), 3))
print('Sin cinturón bien detectado:', round(1 - predichos[reales == 0].mean(), 3))
matriz = np.array([[np.sum((reales == r) & (predichos == p)) for p in (0, 1)] for r in (0, 1)])
plt.imshow(matriz, cmap='Blues')
for r in range(2):
    for p in range(2):
        plt.text(p, r, matriz[r, p], ha='center', va='center', fontsize=20)
plt.xticks([0, 1], ['SIN', 'CON'])
plt.yticks([0, 1], ['SIN', 'CON'])
plt.xlabel('Predicho')
plt.ylabel('Real')
plt.title('Matriz de confusión')
plt.show()

## 10. Fotos de ejemplo
Verde = acertó, rojo = se equivocó.

In [ ]:
from PIL import Image
random.seed(SEMILLA)
indices = random.sample(range(len(prueba)), N_EJEMPLOS)
plt.figure(figsize=(15, 6))
for k, i in enumerate(indices):
    plt.subplot(2, (N_EJEMPLOS + 1) // 2, k + 1)
    plt.imshow(Image.open(prueba[i][0]))
    texto = 'CON' if predichos[i] else 'SIN'
    color = 'green' if predichos[i] == reales[i] else 'red'
    plt.title(f'{texto} cinturón ({probabilidades[i]:.2f})', color=color)
    plt.axis('off')
plt.show()

## 11. Guardar el modelo para la Raspberry Pi

In [ ]:
conversor = tf.lite.TFLiteConverter.from_keras_model(modelo)
conversor.optimizations = [tf.lite.Optimize.DEFAULT]
os.makedirs(f'{CARPETA}/modelo', exist_ok=True)
open(f'{CARPETA}/modelo/modelo_cinturon.tflite', 'wb').write(conversor.convert())
print('Guardado en', f'{CARPETA}/modelo/modelo_cinturon.tflite')